1. Install Dependencies

In [1]:
!pip install transformers datasets seqeval evaluate
!pip install -U datasets

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16282 sha256=29325721a003c2107446071f83ec2fb7f51498f6131f0ee56ac8d02775276983
  Stored in directory: c:\users\nishakart\appdata\local\pip\cache\wheels\14\cf\a7\8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval

   -------------------- ------------------- 1/2 [evaluate]
   -------------------- ------------------- 1/2 [evaluate]
   -------------------- ------------------- 1/2 [evaluate]
   -------------------- ------------------- 1/2 [evaluate]
   ------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 3.2 MB/s  0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2025.12.5 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.4 which is incompatible.
unsloth-zoo 2025.12.4 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.4 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2. Load Dataset

In [33]:
def read_conll(file_path):
    sentences, labels = [], []
    words, tags = [], []

    with open(file_path, "r") as f:
        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
            else:
                parts = line.split()
                words.append(parts[0])
                tags.append(parts[-1])
    return sentences, labels

train_sentences, train_labels_raw = read_conll("train.txt")
val_sentences, val_labels_raw = read_conll("valid.txt")
test_sentences, test_labels_raw = read_conll("test.txt")

# =========================
# 4. Label Mapping
# =========================
unique_labels = sorted(list(set(tag for doc in train_labels_raw for tag in doc)))

label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

def encode_labels(labels):
    return [[label2id[tag] for tag in doc] for doc in labels]

train_labels = encode_labels(train_labels_raw)
val_labels = encode_labels(val_labels_raw)
test_labels = encode_labels(test_labels_raw)

# Set label_list for model and metric setup
label_list = unique_labels

# =========================
# 5. Create Dataset
# =========================
from datasets import Dataset

train_dataset = Dataset.from_dict({"tokens": train_sentences, "ner_tags": train_labels})
val_dataset = Dataset.from_dict({"tokens": val_sentences, "ner_tags": val_labels})
test_dataset = Dataset.from_dict({"tokens": test_sentences, "ner_tags": test_labels})

3. Tokenization + Label Alignment

In [34]:
from transformers import AutoTokenizer
from datasets import DatasetDict

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # ignore
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Create a DatasetDict from the individual datasets
datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

4. Model Setup

In [35]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/_device.py:109: UserWarning: Initializing zero-element tensors is a no-op
  return func(*args, **kwargs)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


5. Training Setup

In [48]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    remove_unused_columns=False
)

6. Evaluation Metric

In [49]:
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_predictions = [
        [label_list[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

7. Trainer

In [50]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

8. Train Model

In [ ]:
trainer.train()

9. Evaluate

In [ ]:
trainer.evaluate()

10. Inference

In [ ]:
import torch

sentence = "John works at Google in California"

inputs = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs).logits
predictions = torch.argmax(outputs, dim=2)

predicted_labels = [label_list[p.item()] for p in predictions[0]]

print(list(zip(sentence.split(), predicted_labels)))